In [4]:
# export_to_onnx.py
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models

CLASS_NAMES = ["house", "not_house"]

# Resolve aimicroservice root for both common launch locations.
cwd = Path.cwd()
if (cwd / "ai_models").exists():
    AIMICRO_ROOT = cwd
elif (cwd / "aimicroservice" / "ai_models").exists():
    AIMICRO_ROOT = cwd / "aimicroservice"
else:
    AIMICRO_ROOT = (cwd / ".." / "..").resolve()

MODEL_PATH = AIMICRO_ROOT / "ai_models" / "house_model_snd.pth"
ONNX_PATH = AIMICRO_ROOT / "ai_models" / "house_model.onnx"

def build_model():
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    return model

def main():
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

    checkpoint = torch.load(MODEL_PATH, map_location="cpu")
    if isinstance(checkpoint, nn.Module):
        model = checkpoint
    else:
        model = build_model()
        model.load_state_dict(checkpoint)

    model.eval()

    dummy = torch.randn(1, 3, 224, 224)
    torch.onnx.export(
        model,
        dummy,
        str(ONNX_PATH),
        input_names=["input"],
        output_names=["logits"],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
        dynamo=False,
    )
    print(f"Exported: {ONNX_PATH}")

if __name__ == "__main__":
    main()

/tmp/ipykernel_33174/1295100105.py:41: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported: /home/marwen/Desktop/WEB_PROJECT/aimicroservice/ai_models/house_model.onnx
